# Reproduce Best Climate-Risk Submission

This notebook reproduces the best confirmed submission using the available training, test, and climate feature files. It fits two balanced Logistic Regression models, blends their probabilities, and writes `best_submission_reproduced.csv`.

Model blend: 70% with `C=2.0` and 30% with `C=1.0`. `TargetF1` cutoff: `0.275`.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from scipy.stats import rankdata

import lightgbm as lgb
from catboost import CatBoostClassifier  # pip install lightgbm catboost

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if not (ROOT / "Train.csv").exists():
    ROOT = ROOT / "Primary competition visual Climate Risk and Health Prediction Challenge"
TARGET, ID = "is_climate_sensitive", "ID"
SEEDS = [42, 7, 2024]
N_FOLDS = 5

train = pd.read_csv(ROOT / "Train.csv")
test = pd.read_csv(ROOT / "Test.csv")
if (ROOT / "climate_features.csv").exists():
    clim = pd.read_csv(ROOT / "climate_features.csv").drop(columns=["deathdate"], errors="ignore")
    clim = clim.drop_duplicates(subset=[ID], keep="last")
    train = train.merge(clim, on=ID, how="left")
    test = test.merge(clim, on=ID, how="left")

y = train[TARGET].astype(int).values
full = pd.concat([train.drop(columns=[TARGET]), test], keys=["tr", "te"])


def has(d, *cols):
    return all(c in d.columns for c in cols)


def make_features(d):
    d = d.copy()
    if "deathdate" in d.columns:
        dt = pd.to_datetime(d["deathdate"], errors="coerce")
        d["death_year"] = dt.dt.year
        d["death_month"] = dt.dt.month
        d["death_dow"] = dt.dt.dayofweek
        d["death_doy"] = dt.dt.dayofyear
        d["death_quarter"] = dt.dt.quarter
        d = d.drop(columns=["deathdate"])
    if has(d, "max_temperature", "min_temperature"):
        d["temp_range"] = d["max_temperature"] - d["min_temperature"]
    if has(d, "age", "avg_temperature"):
        d["age_x_temp"] = d["age"] * d["avg_temperature"]
    if has(d, "age", "elevation"):
        d["age_x_elev"] = d["age"] * d["elevation"]
    if has(d, "avg_temperature", "elevation"):
        d["temp_x_elev"] = d["avg_temperature"] * d["elevation"]
    if has(d, "latitude", "longitude"):
        d["lat_x_lon"] = d["latitude"] * d["longitude"]
        d["lat_lon_dist"] = np.sqrt(d["latitude"] ** 2 + d["longitude"] ** 2)
    if has(d, "rain_sum_30d", "tavg_30d"):
        d["rain_temp_30d"] = d["rain_sum_30d"] / (d["tavg_30d"] + 1)
    if has(d, "tmax_30d", "tmin_30d"):
        d["span_30d"] = d["tmax_30d"] - d["tmin_30d"]
    if has(d, "ndvi_30d", "elevation"):
        d["ndvi_x_elev"] = d["ndvi_30d"] * d["elevation"]
    # frequency encoding for high-cardinality categoricals (computed on train+test)
    for c in ["location", "zone"]:
        if c in d.columns:
            d[c + "_freq"] = d[c].map(d[c].value_counts(dropna=False))
    return d.drop(columns=[ID], errors="ignore")


feat = make_features(full)
feat = feat.loc[:, ~feat.isna().all()]
cat_cols = feat.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in feat.columns if c not in cat_cols]
for c in cat_cols:
    feat[c] = feat[c].fillna("__missing__").astype(str).str.strip()

X, Xt = feat.loc["tr"].reset_index(drop=True), feat.loc["te"].reset_index(drop=True)

# LightGBM version: categoricals as pandas category with shared categories
X_l, Xt_l = X.copy(), Xt.copy()
for c in cat_cols:
    cats = pd.Categorical(feat[c]).categories
    X_l[c] = pd.Categorical(X_l[c], categories=cats)
    Xt_l[c] = pd.Categorical(Xt_l[c], categories=cats)


def lr_model():
    pre = ColumnTransformer([
        ("n", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), num_cols),
        ("c", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])
    return Pipeline([("p", pre), ("m", LogisticRegression(C=1.0, class_weight="balanced",
                                                          solver="liblinear", max_iter=8000))])


def run_model(name, seed, tr_idx, va_idx):
    if name == "lgb":
        m = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.02, num_leaves=31,
                               min_child_samples=30, subsample=0.8, subsample_freq=1,
                               colsample_bytree=0.7, reg_lambda=3.0, random_state=seed,
                               verbose=-1)
        m.fit(X_l.iloc[tr_idx], y[tr_idx], eval_set=[(X_l.iloc[va_idx], y[va_idx])],
              eval_metric="auc", callbacks=[lgb.early_stopping(150, verbose=False)])
        return m.predict_proba(X_l.iloc[va_idx])[:, 1], m.predict_proba(Xt_l)[:, 1]
    if name == "cat":
        m = CatBoostClassifier(iterations=3000, learning_rate=0.03, depth=6, eval_metric="AUC",
                               random_seed=seed, verbose=0, early_stopping_rounds=200,
                               cat_features=cat_cols)
        m.fit(X.iloc[tr_idx], y[tr_idx], eval_set=(X.iloc[va_idx], y[va_idx]))
        return m.predict_proba(X.iloc[va_idx])[:, 1], m.predict_proba(Xt)[:, 1]
    m = lr_model()
    m.fit(X.iloc[tr_idx], y[tr_idx])
    return m.predict_proba(X.iloc[va_idx])[:, 1], m.predict_proba(Xt)[:, 1]


names = ["lgb", "cat", "lr"]
oof = {n: np.zeros(len(X)) for n in names}
pred = {n: np.zeros(len(Xt)) for n in names}

for seed in SEEDS:
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed)
    for tr_idx, va_idx in skf.split(X, y):
        for n in names:
            p_va, p_te = run_model(n, seed, tr_idx, va_idx)
            oof[n][va_idx] += p_va / len(SEEDS)
            pred[n] += p_te / (len(SEEDS) * N_FOLDS)

for n in names:
    print(f"{n}: OOF AUC = {roc_auc_score(y, oof[n]):.5f}")


def rank01(a):
    return (rankdata(a) - 1) / (len(a) - 1)


# blend weight search on OOF (coarse grid, simple to avoid overfitting)
best = (-1, None)
for wl in np.arange(0, 1.01, 0.1):
    for wc in np.arange(0, 1.01 - wl, 0.1):
        wr = 1 - wl - wc
        b = wl * oof["lgb"] + wc * oof["cat"] + wr * oof["lr"]
        s = roc_auc_score(y, b)
        if s > best[0]:
            best = (s, (wl, wc, wr))
auc, (wl, wc, wr) = best
print(f"Best blend weights lgb/cat/lr = {wl:.1f}/{wc:.1f}/{wr:.1f}, OOF AUC = {auc:.5f}")

blend_oof = wl * oof["lgb"] + wc * oof["cat"] + wr * oof["lr"]
blend_te = wl * pred["lgb"] + wc * pred["cat"] + wr * pred["lr"]

# threshold tuned on OOF F1 (fine grid)
ths = np.arange(0.05, 0.80, 0.005)
f1s = [f1_score(y, blend_oof >= t) for t in ths]
thr = ths[int(np.argmax(f1s))]
print(f"Best threshold = {thr:.3f}, OOF F1 = {max(f1s):.5f}")

sub = pd.DataFrame({ID: test[ID], "TargetF1": (blend_te >= thr).astype(int), "TargetRAUC": blend_te})
sub.to_csv(ROOT / "improved_submission.csv", index=False)
print("Saved improved_submission.csv", sub.shape)